In [6]:
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load saved model & exact tokenizer object
model = tf.keras.models.load_model('D:/VishnuMK/Projects/Email/repo/cnn_lstm_phishing_model.keras')

with open('D:/VishnuMK/Projects/Email/repo/tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

def predict_url(url):
    # Tokenize input using loaded tokenizer
    sequences = tokenizer.texts_to_sequences([url.lower().strip()])
    print("Generated integer sequence : ",sequences)
    padded_input = pad_sequences(sequences, maxlen=200, padding='post', truncating='post')
    
    # Predict probability
    prob = model.predict(padded_input, verbose=0)[0][0]
    
    label = "PHISHING ⚠️" if prob > 0.5 else "BENIGN ✅"
    confidence = prob if prob > 0.5 else (1 - prob)
    
    print("-" * 60)
    print(f"URL: {url}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence * 100:.2f}% (Phishing Prob: {prob:.4f})")

# Test custom URL instances
predict_url("https://www.google.com")
predict_url("http://paypal.com-verification-account.login.ru/update")
predict_url("https://github.com/tensorflow/tensorflow")
predict_url("http://amazon.com.security-check-portal.com/auth")

Generated integer sequence :  [[18, 5, 5, 17, 9, 46, 6, 6, 22, 22, 22, 13, 21, 3, 3, 21, 14, 2, 13, 8, 3, 12]]
------------------------------------------------------------
URL: https://www.google.com
Prediction: PHISHING ⚠️
Confidence: 99.85% (Phishing Prob: 0.9985)
Generated integer sequence :  [[18, 5, 5, 17, 46, 6, 6, 17, 4, 26, 17, 4, 14, 13, 8, 3, 12, 15, 31, 2, 10, 7, 23, 7, 8, 4, 5, 7, 3, 11, 15, 4, 8, 8, 3, 19, 11, 5, 13, 14, 3, 21, 7, 11, 13, 10, 19, 6, 19, 17, 16, 4, 5, 2]]
------------------------------------------------------------
URL: http://paypal.com-verification-account.login.ru/update
Prediction: PHISHING ⚠️
Confidence: 99.76% (Phishing Prob: 0.9976)
Generated integer sequence :  [[18, 5, 5, 17, 9, 46, 6, 6, 21, 7, 5, 18, 19, 20, 13, 8, 3, 12, 6, 5, 2, 11, 9, 3, 10, 23, 14, 3, 22, 6, 5, 2, 11, 9, 3, 10, 23, 14, 3, 22]]
------------------------------------------------------------
URL: https://github.com/tensorflow/tensorflow
Prediction: PHISHING ⚠️
Confidence: 99.48% (

In [4]:
import pandas as pd
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

# 1. Load original URLs
raw_csv_path = 'D:/VishnuMK/Projects/Email/URL_detection/malicious_phish.csv/malicious_phish.csv'
df = pd.read_csv(raw_csv_path)
df = df.dropna().drop_duplicates(subset=['url'])
df = df[df['type'].isin(['benign', 'phishing'])]

urls = df['url'].astype(str).values

# 2. FIT TOKENIZER WITH char_level=True AND filters=''
tokenizer = Tokenizer(filters='', lower=True, char_level=True, oov_token='<UNK>')
tokenizer.fit_on_texts(urls)

# Verify sequence length for google.com (MUST BE EXACTLY 22)
test_seq = tokenizer.texts_to_sequences(["https://www.google.com"])
print(f"Generated sequence length: {len(test_seq[0])}")  # Should print 22!
print(f"Sequence: {test_seq[0]}")

# 3. Save to pickle
tokenizer_save_path = 'D:/VishnuMK/Projects/Email/repo/tokenizer.pickle'
with open(tokenizer_save_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Character-level tokenizer successfully saved!")

Generated sequence length: 22
Sequence: [18, 5, 5, 17, 9, 46, 6, 6, 22, 22, 22, 13, 21, 3, 3, 21, 14, 2, 13, 8, 3, 12]
Character-level tokenizer successfully saved!


In [5]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

# 1. Load Tokenizer
with open('D:/VishnuMK/Projects/Email/repo/tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

# 2. Load Data & Tokenize Character-by-Character
raw_csv_path = 'D:/VishnuMK/Projects/Email/URL_detection/malicious_phish.csv/malicious_phish.csv'
df = pd.read_csv(raw_csv_path).dropna().drop_duplicates(subset=['url'])
df = df[df['type'].isin(['benign', 'phishing'])].copy()

# Enforce Benign = 0, Phishing = 1
df['label'] = df['type'].map({'benign': 0, 'phishing': 1})

urls = df['url'].astype(str).values
labels = df['label'].values

MAX_LEN = 200
sequences = tokenizer.texts_to_sequences(urls)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = np.array(labels)

vocab_size = len(tokenizer.word_index) + 2

# 3. Train/Val/Test Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 4. CNN + LSTM Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=MAX_LEN),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_path = 'D:/VishnuMK/Projects/Email/repo/cnn_lstm_phishing_model.keras'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(model_path, monitor='val_loss', save_best_only=True)
]

print("Training CNN+LSTM model...")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=256, callbacks=callbacks)
model.save(model_path)

Training CNN+LSTM model...
Epoch 1/10


d:\VishnuMK\Projects\Email\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1632/1632 ━━━━━━━━━━━━━━━━━━━━ 302s 184ms/step - accuracy: 0.8201 - loss: 0.4734 - val_accuracy: 0.8207 - val_loss: 0.4752
Epoch 2/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 95s 58ms/step - accuracy: 0.8985 - loss: 0.2844 - val_accuracy: 0.9613 - val_loss: 0.1178
Epoch 3/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 97s 59ms/step - accuracy: 0.9662 - loss: 0.1042 - val_accuracy: 0.9723 - val_loss: 0.0861
Epoch 4/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 98s 60ms/step - accuracy: 0.9735 - loss: 0.0840 - val_accuracy: 0.9770 - val_loss: 0.0751
Epoch 5/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.9766 - loss: 0.0751 - val_accuracy: 0.9786 - val_loss: 0.0721
Epoch 6/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 99s 61ms/step - accuracy: 0.9780 - loss: 0.0696 - val_accuracy: 0.9768 - val_loss: 0.0712
Epoch 7/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - accuracy: 0.9799 - loss: 0.0639 - val_accuracy: 0.9793 - val_loss: 0.0618
Epoch 8/10
1632/1632 ━━━━━━━━━━━━━━━━━━━━ 99s 61ms/step - accuracy: 0.9811 - loss: 